##  Instalación y configuración

In [ ]:
# ── Montar Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# Verificar que los archivos existen
import os

CARPETA = "/content/drive/MyDrive/Colab Notebooks/sentimental analisis"

rutas_check = [
    f"{CARPETA}/corpus_tweets_final.csv",
    f"{CARPETA}/resenas_limpio.csv",
]

for ruta in rutas_check:
    if os.path.exists(ruta):
        size = os.path.getsize(ruta) / 1024
        print(f" Encontrado: {os.path.basename(ruta)}  ({size:.1f} KB)")
    else:
        print(f" No encontrado: {ruta}")


In [ ]:
# Instalar dependencias
# pysentimiento incluye RoBERTa-es preentrenado para tweets en español
!pip install pysentimiento torch transformers pandas matplotlib seaborn scikit-learn -q

# Verificar GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo disponible: {device.upper()}")
if device == "cpu":
    print("  Sin GPU — el modelo transformer será más lento (~2-3h para 2.500 tweets)")
    print("   Recomendación: activa GPU en Colab (Entorno → Cambiar tipo de entorno → T4 GPU)")
else:
    print(f" GPU detectada: {torch.cuda.get_device_name(0)}")
    print("   Tiempo estimado: ~15-20 min para 2.500 tweets")


##  Imports y carga del corpus

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import re
import unicodedata
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "axes.titlesize": 12, "axes.titleweight": "bold",
})

AZUL   = "#2E5496"
ROJO   = "#C0392B"
VERDE  = "#27AE60"
NARANJA= "#E67E22"
GRIS   = "#7F8C8D"

# ── Rutas ─────────────────────────────────────────────────────────────────────
# Rutas de los archivos
# Si los tienes en la raíz de Colab, usa solo el nombre del archivo
# Si están en Google Drive, móntalos primero (ver celda opcional abajo)
CARPETA      = "/content/drive/MyDrive/Colab Notebooks/sentimental analisis"
PATH_TWEETS  = f"{CARPETA}/corpus_tweets_final.csv"
PATH_REVIEWS = f"{CARPETA}/resenas_limpio.csv"
OUT_TWEETS   = f"{CARPETA}/corpus_tweets_sentiment.csv"
OUT_REVIEWS  = f"{CARPETA}/resenas_sentiment.csv"

df_tw = pd.read_csv(PATH_TWEETS)
df_rv = pd.read_csv(PATH_REVIEWS)

print(f"Tweets cargados:  {len(df_tw):,}")
print(f"Reseñas cargadas: {len(df_rv):,}")
print(f"\nColumnas tweets: {list(df_tw.columns)}")


---
##  Función de mapeo a 5 clases de intensidad



In [ ]:
def probas_a_5clases(p_pos, p_neg, p_neu, umbral_alto=0.75):
    """
    Convierte probabilidades POS/NEG/NEU a una escala de 5 clases.

    Parámetros:
        p_pos, p_neg, p_neu : probabilidades del modelo (deben sumar ~1)
        umbral_alto         : a partir de qué probabilidad se considera 'muy' (default 0.75)

    Retorna:
        etiqueta (str): MUY_POSITIVO | POSITIVO | NEUTRO | NEGATIVO | MUY_NEGATIVO
        score (float):  valor numérico de -2 a +2 para análisis cuantitativo
    """
    dominante = max([("POS", p_pos), ("NEG", p_neg), ("NEU", p_neu)], key=lambda x: x[1])[0]

    if dominante == "POS":
        if p_pos >= umbral_alto:
            return "MUY_POSITIVO", 2.0
        else:
            return "POSITIVO", 1.0
    elif dominante == "NEG":
        if p_neg >= umbral_alto:
            return "MUY_NEGATIVO", -2.0
        else:
            return "NEGATIVO", -1.0
    else:
        return "NEUTRO", 0.0

# Mapeo numérico para cálculos
SCORE_MAP = {"MUY_POSITIVO": 2, "POSITIVO": 1, "NEUTRO": 0, "NEGATIVO": -1, "MUY_NEGATIVO": -2}
ORDEN_CLASES = ["MUY_POSITIVO", "POSITIVO", "NEUTRO", "NEGATIVO", "MUY_NEGATIVO"]
COLORES_SA = {
    "MUY_POSITIVO": "#1a5276",
    "POSITIVO":     "#2980b9",
    "NEUTRO":       "#95a5a6",
    "NEGATIVO":     "#e74c3c",
    "MUY_NEGATIVO": "#922b21",
}

print("Función de mapeo a 5 clases lista.")
print("Ejemplo:")
print(f"  POS=0.85, NEG=0.10, NEU=0.05 → {probas_a_5clases(0.85, 0.10, 0.05)}")
print(f"  POS=0.60, NEG=0.30, NEU=0.10 → {probas_a_5clases(0.60, 0.30, 0.10)}")
print(f"  POS=0.10, NEG=0.80, NEU=0.10 → {probas_a_5clases(0.10, 0.80, 0.10)}")


---
##  Enfoque 1: Análisis de sentimiento basado en léxico




In [ ]:
import urllib.request

# Descargar ML-SentiCon si no está disponible localmente
# Fuente: https://timm.ujaen.es/recursos/ml-senticon/
LEXICON_URL = "https://raw.githubusercontent.com/laugustyniak/abusive-language-pl/master/data/sentiment-lexicon-es.txt"

# Léxico manual de respaldo con las palabras más relevantes para el corpus UAM
# (términos identificados en el análisis descriptivo con carga emocional clara)
LEXICO_MANUAL = {
    # Muy positivo
    "excelente": 0.9, "increíble": 0.85, "genial": 0.85, "perfecto": 0.9,
    "extraordinario": 0.9, "brillante": 0.85, "fantástico": 0.85,
    "maravilloso": 0.85, "espectacular": 0.85, "impresionante": 0.8,
    "encantado": 0.8, "encantada": 0.8, "satisfecho": 0.7, "satisfecha": 0.7,
    "recomendable": 0.75, "recomiendo": 0.7, "felicito": 0.75,
    "felicitaciones": 0.75, "enhorabuena": 0.6, "orgulloso": 0.7,
    "mejor": 0.5, "bueno": 0.5, "buena": 0.5, "bien": 0.4,
    # Muy negativo
    "pésimo": -0.9, "horrible": -0.9, "fatal": -0.85, "nefasto": -0.85,
    "catastrófico": -0.9, "vergonzoso": -0.85, "vergüenza": -0.8,
    "desastre": -0.8, "decepcionante": -0.75, "decepción": -0.75,
    "lamentable": -0.8, "inaceptable": -0.8, "indignante": -0.8,
    "injusto": -0.7, "injusta": -0.7, "deplorable": -0.85,
    "harto": -0.6, "harta": -0.6, "arrepiento": -0.7,
    "malo": -0.5, "mala": -0.5, "mal": -0.4, "peor": -0.6,
    "problema": -0.3, "problemas": -0.3, "queja": -0.4,
    # Palabras académicas con carga negativa en contexto universitario
    "suspenso": -0.4, "suspensos": -0.4, "cateo": -0.5, "cateado": -0.5,
    "denegada": -0.5, "denegado": -0.5, "rechazado": -0.5, "rechazada": -0.5,
    "discriminación": -0.7, "acoso": -0.8, "huelga": -0.3,
    # Palabras académicas con carga positiva
    "aprobado": 0.4, "aprobada": 0.4, "matrícula": 0.1,
    "excelencia": 0.7, "calidad": 0.5, "reconocimiento": 0.5,
}

def analizar_lexico(texto, lexico=LEXICO_MANUAL, umbral_alto=0.75):
    """
    Aplica análisis de sentimiento basado en léxico a un texto.
    Retorna etiqueta de 5 clases, score numérico y palabras clave encontradas.
    """
    # Normalizar texto
    t = str(texto).lower()
    t = ''.join(c for c in unicodedata.normalize('NFD', t)
                if unicodedata.category(c) != 'Mn')
    tokens = re.findall(r'\b[a-z]{3,}\b', t)

    # Buscar tokens en léxico
    scores_encontrados = []
    palabras_clave = []
    for tok in tokens:
        if tok in lexico:
            scores_encontrados.append(lexico[tok])
            palabras_clave.append((tok, lexico[tok]))

    if not scores_encontrados:
        return "NEUTRO", 0.0, []

    score_medio = np.mean(scores_encontrados)

    # Convertir score [-1,+1] a probabilidades aproximadas para usar probas_a_5clases
    if score_medio > 0:
        p_pos = min(0.5 + score_medio * 0.5, 0.99)
        p_neg = (1 - p_pos) * 0.2
        p_neu = 1 - p_pos - p_neg
    elif score_medio < 0:
        p_neg = min(0.5 + abs(score_medio) * 0.5, 0.99)
        p_pos = (1 - p_neg) * 0.2
        p_neu = 1 - p_pos - p_neg
    else:
        p_neu, p_pos, p_neg = 0.7, 0.15, 0.15

    etiqueta, _ = probas_a_5clases(p_pos, p_neg, p_neu, umbral_alto)
    return etiqueta, round(score_medio, 4), palabras_clave

# Test
ejemplos = [
    "Esta universidad es excelente, los profesores son fantásticos",
    "El examen fue muy difícil y creo que he suspendido",
    "Mañana tengo clase de psicología en la Autónoma",
    "Vergonzoso el trato en la secretaría, inaceptable",
]
print("Test del analizador léxico:")
for ej in ejemplos:
    et, sc, kw = analizar_lexico(ej)
    print(f"  [{et:15s}] score={sc:+.2f} | {ej[:60]}")
    if kw:
        print(f"             palabras clave: {kw}")


### Aplicar léxico al corpus completo

In [ ]:
print("Aplicando análisis léxico a tweets...")
resultados_lex_tw = [analizar_lexico(t) for t in df_tw["text_clean"]]
df_tw["lex_label"]  = [r[0] for r in resultados_lex_tw]
df_tw["lex_score"]  = [r[1] for r in resultados_lex_tw]
df_tw["lex_kwords"] = [str(r[2]) for r in resultados_lex_tw]

print("Aplicando análisis léxico a reseñas...")
resultados_lex_rv = [analizar_lexico(t) for t in df_rv["text_clean"].fillna("")]
df_rv["lex_label"] = [r[0] for r in resultados_lex_rv]
df_rv["lex_score"] = [r[1] for r in resultados_lex_rv]

print("\nDistribución léxico — tweets:")
print(df_tw["lex_label"].value_counts())
print("\nDistribución léxico — reseñas:")
print(df_rv["lex_label"].value_counts())


---
##  Enfoque 2: Análisis de sentimiento con RoBERTa-es (pysentimiento)




In [ ]:
from pysentimiento import create_analyzer

print("Cargando modelo RoBERTa-es...")
analyzer = create_analyzer(task="sentiment", lang="es")
print(f"Modelo cargado. Dispositivo: {device.upper()}")

# Test del modelo
print("\nTest del modelo transformer:")
for ej in ejemplos:
    result = analyzer.predict(ej)
    et, sc = probas_a_5clases(
        result.probas.get("POS", 0),
        result.probas.get("NEG", 0),
        result.probas.get("NEU", 0)
    )
    print(f"  [{et:15s}] score={sc:+.1f} | prob={result.probas} | {ej[:55]}")


In [ ]:
from tqdm.auto import tqdm

def analizar_transformer(textos, batch_size=32):
    """
    Aplica el modelo RoBERTa-es a una lista de textos en lotes.
    Retorna lista de (etiqueta_5clases, score, p_pos, p_neg, p_neu).
    """
    resultados = []
    for i in tqdm(range(0, len(textos), batch_size), desc="Analizando"):
        lote = textos[i:i+batch_size]
        for texto in lote:
            try:
                r = analyzer.predict(str(texto)[:512])  # límite de tokens
                p_pos = r.probas.get("POS", 0)
                p_neg = r.probas.get("NEG", 0)
                p_neu = r.probas.get("NEU", 0)
                et, sc = probas_a_5clases(p_pos, p_neg, p_neu)
                resultados.append((et, sc, round(p_pos,4), round(p_neg,4), round(p_neu,4)))
            except Exception as e:
                resultados.append(("NEUTRO", 0.0, 0.0, 0.0, 1.0))
    return resultados

print("Aplicando RoBERTa-es a tweets...")
res_tw = analizar_transformer(df_tw["text_clean"].tolist())
df_tw["rob_label"] = [r[0] for r in res_tw]
df_tw["rob_score"] = [r[1] for r in res_tw]
df_tw["rob_p_pos"] = [r[2] for r in res_tw]
df_tw["rob_p_neg"] = [r[3] for r in res_tw]
df_tw["rob_p_neu"] = [r[4] for r in res_tw]

print("\nAplicando RoBERTa-es a reseñas...")
res_rv = analizar_transformer(df_rv["text_clean"].fillna("").tolist())
df_rv["rob_label"] = [r[0] for r in res_rv]
df_rv["rob_score"] = [r[1] for r in res_rv]
df_rv["rob_p_pos"] = [r[2] for r in res_rv]
df_rv["rob_p_neg"] = [r[3] for r in res_rv]
df_rv["rob_p_neu"] = [r[4] for r in res_rv]

print("\nDistribución RoBERTa — tweets:")
print(df_tw["rob_label"].value_counts())
print("\nDistribución RoBERTa — reseñas:")
print(df_rv["rob_label"].value_counts())


---
##  Validación contra ground truth (reseñas Google)




In [ ]:
# Convertir etiquetas de 5 a 3 clases para comparación
def a_3clases(label_5):
    if label_5 in ("MUY_POSITIVO", "POSITIVO"):
        return "POSITIVO"
    elif label_5 in ("MUY_NEGATIVO", "NEGATIVO"):
        return "NEGATIVO"
    else:
        return "NEUTRO"

# Ground truth: sentimiento_proxy de las reseñas (ya calculado en limpieza)
df_rv_eval = df_rv.dropna(subset=["sentimiento_proxy"]).copy()
df_rv_eval["gt"] = df_rv_eval["sentimiento_proxy"].str.upper()

df_rv_eval["lex_3"] = df_rv_eval["lex_label"].apply(a_3clases)
df_rv_eval["rob_3"] = df_rv_eval["rob_label"].apply(a_3clases)

print(f"Reseñas con ground truth: {len(df_rv_eval)}")
print(f"\nDistribución ground truth:\n{df_rv_eval['gt'].value_counts()}")

# Métricas enfoque léxico
print("\n" + "="*55)
print("ENFOQUE LÉXICO vs Ground Truth")
print("="*55)
print(classification_report(df_rv_eval["gt"], df_rv_eval["lex_3"],
                             labels=["POSITIVO","NEUTRO","NEGATIVO"]))
kappa_lex = cohen_kappa_score(df_rv_eval["gt"], df_rv_eval["lex_3"])
print(f"Cohen's Kappa (léxico): {kappa_lex:.3f}")

# Métricas enfoque transformer
print("\n" + "="*55)
print("ENFOQUE ROBERTA vs Ground Truth")
print("="*55)
print(classification_report(df_rv_eval["gt"], df_rv_eval["rob_3"],
                             labels=["POSITIVO","NEUTRO","NEGATIVO"]))
kappa_rob = cohen_kappa_score(df_rv_eval["gt"], df_rv_eval["rob_3"])
print(f"Cohen's Kappa (RoBERTa): {kappa_rob:.3f}")

print(f"\n→ Diferencia Kappa: {kappa_rob - kappa_lex:+.3f} a favor de {'RoBERTa' if kappa_rob > kappa_lex else 'Léxico'}")


In [ ]:
# Matrices de confusión comparativas
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor="white")
clases = ["POSITIVO", "NEUTRO", "NEGATIVO"]

for ax, (nombre, pred, kappa) in zip(axes, [
    ("Enfoque Léxico", df_rv_eval["lex_3"], kappa_lex),
    ("RoBERTa-es",     df_rv_eval["rob_3"], kappa_rob),
]):
    ax.set_facecolor("white")
    cm = confusion_matrix(df_rv_eval["gt"], pred, labels=clases)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=clases, yticklabels=clases,
                ax=ax, cbar=False, linewidths=0.5, linecolor="#DDDDDD",
                annot_kws={"size": 10})
    ax.set_xlabel("Predicción", fontsize=10, color="#222222")
    ax.set_ylabel("Ground truth (puntuación Google)", fontsize=10, color="#222222")
    ax.set_title(f"{nombre}\nCohen's κ = {kappa:.3f}", fontsize=11,
                 fontweight="bold", color="#222222")
    ax.tick_params(colors="#222222")

fig.suptitle("Fig. 8 — Validación de enfoques contra ground truth (reseñas Google)",
             fontsize=12, fontweight="bold", color="#222222", y=1.02)
plt.tight_layout()
plt.savefig("fig8_validacion_modelos.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()

print("\n Interpretación de Cohen's Kappa:")
print("   κ < 0.20  → acuerdo pobre")
print("   κ 0.20–0.40 → acuerdo débil")
print("   κ 0.40–0.60 → acuerdo moderado")
print("   κ 0.60–0.80 → acuerdo sustancial")
print("   κ > 0.80  → acuerdo casi perfecto")


---
## Selección del modelo principal



In [ ]:
# Selección automática basada en Kappa
if kappa_rob >= kappa_lex:
    modelo_principal = "RoBERTa-es"
    df_tw["sentiment_label"] = df_tw["rob_label"]
    df_tw["sentiment_score"] = df_tw["rob_score"]
    df_rv["sentiment_label"] = df_rv["rob_label"]
    df_rv["sentiment_score"] = df_rv["rob_score"]
else:
    modelo_principal = "Léxico"
    df_tw["sentiment_label"] = df_tw["lex_label"]
    df_tw["sentiment_score"] = df_tw["lex_score"]
    df_rv["sentiment_label"] = df_rv["lex_label"]
    df_rv["sentiment_score"] = df_rv["lex_score"]

df_tw["sentiment_score_num"] = df_tw["sentiment_label"].map(SCORE_MAP)
df_rv["sentiment_score_num"] = df_rv["sentiment_label"].map(SCORE_MAP)

print(f" Modelo principal seleccionado: {modelo_principal}")
print(f"   (κ RoBERTa={kappa_rob:.3f}  |  κ Léxico={kappa_lex:.3f})")
print(f"\nDistribución final de sentimiento — tweets:")
print(df_tw["sentiment_label"].value_counts().reindex(ORDEN_CLASES))


---
##  Distribución global de sentimiento

###  Comparación de los dos enfoques


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")

for ax, (col, titulo) in zip(axes, [
    ("lex_label",  "Enfoque Léxico"),
    ("rob_label",  "Enfoque RoBERTa-es"),
]):
    ax.set_facecolor("white")
    counts = df_tw[col].value_counts().reindex(ORDEN_CLASES).fillna(0)
    colores = [COLORES_SA[c] for c in ORDEN_CLASES]
    bars = ax.bar(ORDEN_CLASES, counts.values, color=colores,
                  width=0.6, zorder=3, edgecolor="white")
    ax.set_title(titulo, fontsize=11, fontweight="bold", color="#222222")
    ax.set_ylabel("Nº de tweets", color="#222222", fontsize=10)
    ax.set_xticklabels([c.replace("_", "\n") for c in ORDEN_CLASES],
                       fontsize=9, color="#222222")
    ax.tick_params(colors="#222222")
    ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
    ax.grid(axis="y", alpha=0.25, zorder=1)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 10, f"{int(val)}\n({val/len(df_tw)*100:.1f}%)",
                ha="center", va="bottom", fontsize=8, color="#222222")

fig.suptitle("Fig. 9 — Distribución de sentimiento: comparación de enfoques (tweets)",
             fontsize=12, fontweight="bold", color="#222222", y=1.02)
plt.tight_layout()
plt.savefig("fig9_distribucion_global.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


### Tweets vs Reseñas Google

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")

for ax, (df_plot, titulo, n) in zip(axes, [
    (df_tw, f"Tweets (n={len(df_tw):,})", len(df_tw)),
    (df_rv, f"Reseñas Google (n={len(df_rv):,})", len(df_rv)),
]):
    ax.set_facecolor("white")
    counts = df_plot["sentiment_label"].value_counts().reindex(ORDEN_CLASES).fillna(0)
    colores = [COLORES_SA[c] for c in ORDEN_CLASES]
    bars = ax.bar(ORDEN_CLASES, counts.values, color=colores,
                  width=0.6, zorder=3, edgecolor="white")
    ax.set_title(titulo, fontsize=11, fontweight="bold", color="#222222")
    ax.set_ylabel("Nº de registros", color="#222222", fontsize=10)
    ax.set_xticklabels([c.replace("_", "\n") for c in ORDEN_CLASES],
                       fontsize=9, color="#222222")
    ax.tick_params(colors="#222222")
    ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
    ax.grid(axis="y", alpha=0.25, zorder=1)
    for bar, val in zip(bars, counts.values):
        pct = val/n*100
        ax.text(bar.get_x() + bar.get_width()/2, val + 2,
                f"{int(val)}\n({pct:.1f}%)",
                ha="center", va="bottom", fontsize=8, color="#222222")

fig.suptitle("Fig. 10 — Distribución de sentimiento: tweets vs reseñas Google",
             fontsize=12, fontweight="bold", color="#222222", y=1.02)
plt.tight_layout()
plt.savefig("fig10_tweets_vs_resenas.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


---
##  Evolución temporal del sentimiento




In [ ]:
df_tw["año_mes"] = df_tw["año_mes"].astype(str)
evol = df_tw.groupby("año_mes").agg(
    score_medio=("sentiment_score_num", "mean"),
    n_tweets=("sentiment_score_num", "count"),
    pct_pos=("sentiment_label", lambda x: (x.isin(["POSITIVO","MUY_POSITIVO"])).mean() * 100),
    pct_neg=("sentiment_label", lambda x: (x.isin(["NEGATIVO","MUY_NEGATIVO"])).mean() * 100),
).reset_index().sort_values("año_mes")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), facecolor="white",
                                 gridspec_kw={"height_ratios": [2, 1]})

# Panel superior: score medio mensual
ax1.set_facecolor("white")
colores_evol = [ROJO if s < 0 else (AZUL if s > 0.2 else GRIS)
                for s in evol["score_medio"]]
bars = ax1.bar(range(len(evol)), evol["score_medio"],
               color=colores_evol, width=0.75, zorder=3, edgecolor="white", alpha=0.8)

# Media móvil 3 meses
mm = evol["score_medio"].rolling(3, center=True).mean()
ax1.plot(range(len(evol)), mm, color="#333333", linewidth=2,
         linestyle="--", zorder=4, label="Media móvil (3 meses)")
ax1.axhline(0, color="#AAAAAA", linewidth=1, linestyle=":", zorder=2)

# Línea de media global
media_g = evol["score_medio"].mean()
ax1.axhline(media_g, color=AZUL, linewidth=1.2, linestyle=":",
            label=f"Media global ({media_g:+.2f})", alpha=0.6)

etiq_x = [m if m.endswith("-01") or m.endswith("-06") or m.endswith("-09")
          else "" for m in evol["año_mes"]]
ax1.set_xticks(range(len(evol)))
ax1.set_xticklabels(etiq_x, rotation=45, ha="right", fontsize=8.5, color="#222222")
ax1.set_ylabel("Score medio de sentimiento\n(-2 = muy neg. → +2 = muy pos.)",
               color="#222222", fontsize=10)
ax1.set_title("Fig. 11 — Evolución temporal del sentimiento sobre la UAM Madrid (2023–2026)",
              fontsize=12, fontweight="bold", color="#222222")
ax1.legend(fontsize=9, frameon=True, facecolor="white", edgecolor="#CCCCCC")
ax1.grid(axis="y", alpha=0.25, zorder=1)
ax1.tick_params(colors="#222222")
ax1.spines["left"].set_color("#DDDDDD"); ax1.spines["bottom"].set_color("#DDDDDD")

# Panel inferior: % positivo vs negativo apilado
ax2.set_facecolor("white")
ax2.bar(range(len(evol)), evol["pct_pos"], color=AZUL, width=0.75,
        label="% Positivo+Muy positivo", zorder=3, alpha=0.8)
ax2.bar(range(len(evol)), -evol["pct_neg"], color=ROJO, width=0.75,
        label="% Negativo+Muy negativo", zorder=3, alpha=0.8)
ax2.axhline(0, color="#AAAAAA", linewidth=0.8)
ax2.set_xticks(range(len(evol)))
ax2.set_xticklabels(etiq_x, rotation=45, ha="right", fontsize=8.5, color="#222222")
ax2.set_ylabel("% tweets", color="#222222", fontsize=9)
ax2.legend(fontsize=8, frameon=True, facecolor="white", edgecolor="#CCCCCC")
ax2.grid(axis="y", alpha=0.25, zorder=1)
ax2.tick_params(colors="#222222")
ax2.spines["left"].set_color("#DDDDDD"); ax2.spines["bottom"].set_color("#DDDDDD")

plt.tight_layout()
plt.savefig("fig11_evolucion_temporal.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()

print("Estadísticas de evolución temporal:")
print(f"  Mes más positivo:  {evol.loc[evol['score_medio'].idxmax(), 'año_mes']} "
      f"(score={evol['score_medio'].max():+.3f})")
print(f"  Mes más negativo:  {evol.loc[evol['score_medio'].idxmin(), 'año_mes']} "
      f"(score={evol['score_medio'].min():+.3f})")
print(f"  Score global medio: {media_g:+.3f}")


---
##  Sentimiento por categoría semántica




In [ ]:
# Añadir categoría semántica
MAPA_CAT = {
    "mention_query": "Mención\ninstitucional",
    "university_name": "Mención\ninstitucional",
    "student_problems": "Momentos\nacadémicos",
    "momentos_academicos": "Momentos\nacadémicos",
    "momentos_academicos_madrid": "Momentos\nacadémicos",
    "experiencia_directa": "Experiencia\ny valoración",
    "experiencia_uam_madrid": "Experiencia\ny valoración",
    "valoracion_explicita": "Experiencia\ny valoración",
    "valoracion_uam_madrid": "Experiencia\ny valoración",
}
df_tw["categoria"] = df_tw["query_type"].map(MAPA_CAT)

# Score medio por categoría
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")

# Panel 1: distribución de 5 clases por categoría
cat_data = df_tw.groupby(["categoria", "sentiment_label"]).size().unstack(fill_value=0)
cat_data_pct = cat_data.div(cat_data.sum(axis=1), axis=0) * 100
cat_data_pct = cat_data_pct.reindex(columns=ORDEN_CLASES, fill_value=0)

ax = axes[0]
ax.set_facecolor("white")
bottom = np.zeros(len(cat_data_pct))
for clase in ORDEN_CLASES:
    vals = cat_data_pct[clase].values if clase in cat_data_pct.columns else np.zeros(len(cat_data_pct))
    ax.bar(range(len(cat_data_pct)), vals, bottom=bottom,
           color=COLORES_SA[clase], label=clase.replace("_", " "),
           width=0.6, edgecolor="white")
    bottom += vals

ax.set_xticks(range(len(cat_data_pct)))
ax.set_xticklabels(cat_data_pct.index, fontsize=9.5, color="#222222")
ax.set_ylabel("% de tweets", color="#222222", fontsize=10)
ax.set_title("Distribución de sentimiento\npor categoría (%)", fontsize=11,
             fontweight="bold", color="#222222")
ax.legend(fontsize=8, loc="upper right", frameon=True,
          facecolor="white", edgecolor="#CCCCCC", labelcolor="#222222")
ax.grid(axis="y", alpha=0.25, zorder=1)
ax.tick_params(colors="#222222")
ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")

# Panel 2: score medio por categoría
scores_cat = df_tw.groupby("categoria")["sentiment_score_num"].agg(["mean","sem"]).reset_index()
ax2 = axes[1]
ax2.set_facecolor("white")
colores_sc = [ROJO if m < 0 else AZUL for m in scores_cat["mean"]]
bars2 = ax2.bar(range(len(scores_cat)), scores_cat["mean"],
                yerr=scores_cat["sem"] * 1.96,
                color=colores_sc, width=0.5, zorder=3,
                edgecolor="white", capsize=5, error_kw={"color":"#555555","lw":1.5})
ax2.axhline(0, color="#AAAAAA", linewidth=0.8, linestyle=":")
ax2.set_xticks(range(len(scores_cat)))
ax2.set_xticklabels(scores_cat["categoria"], fontsize=9.5, color="#222222")
ax2.set_ylabel("Score medio de sentimiento\n(IC 95%)", color="#222222", fontsize=10)
ax2.set_title("Score medio por categoría\n(barras de error = IC 95%)",
              fontsize=11, fontweight="bold", color="#222222")
for i, (_, row) in enumerate(scores_cat.iterrows()):
    ax2.text(i, row["mean"] + 0.03, f"{row['mean']:+.2f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold", color="#222222")
ax2.grid(axis="y", alpha=0.25, zorder=1)
ax2.tick_params(colors="#222222")
ax2.spines["left"].set_color("#DDDDDD"); ax2.spines["bottom"].set_color("#DDDDDD")

fig.suptitle("Fig. 12 — Sentimiento por categoría semántica de query",
             fontsize=12, fontweight="bold", color="#222222", y=1.02)
plt.tight_layout()
plt.savefig("fig12_sentimiento_categoria.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


---
##  Ejemplos cualitativos: tweets más extremos



In [ ]:
print("=" * 65)
print("TWEETS MÁS POSITIVOS (MUY_POSITIVO con mayor probabilidad)")
print("=" * 65)
top_pos = df_tw[df_tw["sentiment_label"] == "MUY_POSITIVO"].nlargest(8, "rob_p_pos")
for i, (_, row) in enumerate(top_pos.iterrows(), 1):
    print(f"\n{i}. [p_pos={row.get('rob_p_pos', 'N/A'):.2f}] {row['text'][:200]}")

print("\n" + "=" * 65)
print("TWEETS MÁS NEGATIVOS (MUY_NEGATIVO con mayor probabilidad)")
print("=" * 65)
top_neg = df_tw[df_tw["sentiment_label"] == "MUY_NEGATIVO"].nlargest(8, "rob_p_neg")
for i, (_, row) in enumerate(top_neg.iterrows(), 1):
    print(f"\n{i}. [p_neg={row.get('rob_p_neg', 'N/A'):.2f}] {row['text'][:200]}")


---
## Preparación para el cruce con Topic Modelling (ABSA)




In [ ]:
# Columnas que se exportan para el cruce posterior con LDA
COLS_EXPORT = [
    "tweet_id", "fecha", "año", "mes", "año_mes",
    "text", "text_clean",
    "query_type", "version", "categoria",
    "lex_label", "lex_score",
    "rob_label", "rob_score", "rob_p_pos", "rob_p_neg", "rob_p_neu",
    "sentiment_label", "sentiment_score_num",
    "likes", "retweets", "views",
]

# Filtrar solo columnas que existen
cols_disponibles = [c for c in COLS_EXPORT if c in df_tw.columns]
df_export = df_tw[cols_disponibles].copy()

df_export.to_csv(OUT_TWEETS, index=False, encoding="utf-8-sig")
df_rv.to_csv(OUT_REVIEWS, index=False, encoding="utf-8-sig")

print(f" corpus_tweets_sentiment.csv  → {len(df_export):,} tweets")
print(f" resenas_sentiment.csv        → {len(df_rv):,} reseñas")
print(f"\nFiguras guardadas:")
for fig in ["fig8_validacion_modelos.png","fig9_distribucion_global.png",
            "fig10_tweets_vs_resenas.png","fig11_evolucion_temporal.png",
            "fig12_sentimiento_categoria.png"]:
    print(f"  {fig}")
print(f"\n Siguiente paso: Topic Modelling (LDA)")
print(f"   Input: corpus_tweets_sentiment.csv")
print(f"   Output: corpus con columna 'topic_id' para cruzar con sentimiento")
